## Feature Engineering

### Customer Level Features

In [0]:
df = spark.sql("""
SELECT *
FROM resume_project.gold_layer.gold_master
""")
display(df)

sales_id,quantity,discount,total_amount,date,customer_id,name,email,email_domain,location,signup_date,product_id,product_name,category,price,store_id,store_name,region
68,1,19.53,83.56,2025-06-12,81,stephanie foster,hcantu@parker.com,parker,port lisaville,2024-10-11,27,Also Term,Clothing,103.84,5,Store_5,EAST
73,2,21.77,83.47,2025-03-12,54,christopher williams,ejones@owens.info,owens,east rebecca,2023-08-18,38,Official Reality,Clothing,53.35,14,Store_14,NORTH
84,1,24.3,237.34,2025-04-06,68,david kennedy,woodsjamie@yahoo.com,yahoo,gloverstad,2025-02-28,82,Significant Them,Toys,313.53,18,Store_18,WEST
110,4,14.44,1386.62,2025-01-06,148,jeffrey henson,alexanderwagner@yahoo.com,yahoo,lake tammy,2025-02-28,36,Republican Will,Electronics,405.16,8,Store_8,WEST
120,3,14.95,785.53,2025-02-10,129,bruce newman,diazstephanie@hotmail.com,hotmail,north stacy,2023-10-17,21,Want Value,Furniture,307.87,14,Store_14,NORTH
130,1,0.43,480.91,2024-08-08,90,jeffrey fisher,eddie87@miller.biz,miller,lewistown,2023-12-16,35,School Support,Toys,482.99,16,Store_16,NORTH
137,3,29.01,71.88,2025-05-01,140,catherine lopez,samuel81@herrera.org,herrera,south eugeneview,2025-05-06,7,Today Want,Clothing,33.75,2,Store_2,WEST
155,4,24.24,903.82,2025-06-06,200,danielle parker,natalie37@burns.org,burns,duncanfurt,2023-10-02,29,Player Type,Electronics,298.25,12,Store_12,EAST
157,1,15.44,62.62,2024-09-12,166,pamela nguyen,darrenjacobson@yahoo.com,yahoo,millerstad,2022-11-13,22,Any Very,Books,74.05,15,Store_15,WEST
163,1,0.01,432.2,2024-12-02,93,jeffrey schultz,sandracarpenter@neal.com,neal,east ericview,2025-03-16,81,Summer Peace,Furniture,432.24,12,Store_12,EAST


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
##days since last purchase
cust_summary = df.groupBy("customer_id").agg(F.max("date").alias("last_order_date"))


cust_summary = cust_summary.withColumn("days_since_last_purchase", F.datediff(F.current_date(), "last_order_date"))
display(cust_summary)


customer_id,last_order_date,days_since_last_purchase
81,2025-06-28,290
54,2025-06-28,290
68,2025-06-28,290
148,2025-06-28,290
129,2025-06-28,290
90,2025-06-28,290
140,2025-06-28,290
200,2025-06-28,290
166,2025-06-28,290
93,2025-06-28,290


In [0]:
##number of purchases
cust_orders = df.groupBy("customer_id").agg(F.countDistinct("sales_id").alias("num_orders"))
display(cust_orders)
cust_summary = cust_summary.join(cust_orders, on="customer_id", how="inner")


customer_id,num_orders
81,10
54,15
68,9
148,11
129,14
90,16
140,12
200,10
166,10
93,11


In [0]:
##total spend
cust_rev = df.groupBy("customer_id").agg(F.round(F.sum("total_amount"), 2).alias("total_revenue"))
cust_summary = cust_summary.join(cust_rev, on="customer_id", how="inner")
display(cust_summary)

customer_id,last_order_date,days_since_last_purchase,num_orders,total_revenue
93,2025-06-28,290,11,4887.82
186,2025-06-28,290,10,6405.15
38,2025-06-28,290,11,6052.09
190,2025-06-28,290,15,7202.3
12,2025-06-28,290,14,4662.48
67,2025-06-28,290,7,2477.03
161,2025-06-28,290,14,6232.67
70,2025-06-28,290,11,5338.2
18,2025-06-28,290,6,3312.89
105,2025-06-28,290,8,4757.27


In [0]:
##average spend
avg_cust_rev = df.groupBy("customer_id").agg(F.round(F.avg("total_amount"), 2).alias("avg_revenue"))
cust_summary = cust_summary.join(avg_cust_rev, on="customer_id", how="inner")
display(cust_summary)

customer_id,last_order_date,days_since_last_purchase,num_orders,total_revenue,avg_revenue
93,2025-06-28,290,11,4887.82,444.35
186,2025-06-28,290,10,6405.15,640.52
38,2025-06-28,290,11,6052.09,550.19
190,2025-06-28,290,15,7202.3,480.15
12,2025-06-28,290,14,4662.48,333.03
67,2025-06-28,290,7,2477.03,353.86
161,2025-06-28,290,14,6232.67,445.19
70,2025-06-28,290,11,5338.2,485.29
18,2025-06-28,290,6,3312.89,552.15
105,2025-06-28,290,8,4757.27,594.66


In [0]:
##total quantity purchased
total_qut =  df.groupBy("customer_id").agg(F.round(F.sum("quantity"), 2).alias("total_quantity_purchased"))
cust_summary = cust_summary.join(total_qut, on="customer_id", how="inner")
display(cust_summary)

customer_id,last_order_date,days_since_last_purchase,num_orders,total_revenue,avg_revenue,total_quantity_purchased
93,2025-06-28,290,11,4887.82,444.35,25
186,2025-06-28,290,10,6405.15,640.52,26
38,2025-06-28,290,11,6052.09,550.19,25
190,2025-06-28,290,15,7202.3,480.15,39
12,2025-06-28,290,14,4662.48,333.03,29
67,2025-06-28,290,7,2477.03,353.86,16
161,2025-06-28,290,14,6232.67,445.19,34
70,2025-06-28,290,11,5338.2,485.29,29
18,2025-06-28,290,6,3312.89,552.15,13
105,2025-06-28,290,8,4757.27,594.66,19


In [0]:
##number of distinct products purchased
distinct_prod = df.groupBy("customer_id").agg(F.countDistinct("product_id").alias("unique_products_purchased"))
cust_summary = cust_summary.join(distinct_prod, on="customer_id", how="inner")
display(cust_summary)

customer_id,last_order_date,days_since_last_purchase,num_orders,total_revenue,avg_revenue,total_quantity_purchased,unique_products_purchased
93,2025-06-28,290,11,4887.82,444.35,25,10
190,2025-06-28,290,15,7202.3,480.15,39,14
12,2025-06-28,290,14,4662.48,333.03,29,14
70,2025-06-28,290,11,5338.2,485.29,29,11
161,2025-06-28,290,14,6232.67,445.19,34,13
186,2025-06-28,290,10,6405.15,640.52,26,9
38,2025-06-28,290,11,6052.09,550.19,25,11
18,2025-06-28,290,6,3312.89,552.15,13,6
67,2025-06-28,290,7,2477.03,353.86,16,7
141,2025-06-28,290,12,5519.4,459.95,32,11


In [0]:
##number of distinct categories purchased
distinct_cat = df.groupBy("customer_id").agg(F.countDistinct("category").alias("unique_categories_purchased"))
cust_summary = cust_summary.join(distinct_cat, on="customer_id", how="inner")
display(cust_summary)

customer_id,last_order_date,days_since_last_purchase,num_orders,total_revenue,avg_revenue,total_quantity_purchased,unique_products_purchased,unique_categories_purchased
38,2025-06-28,290,11,6052.09,550.19,25,11,5
93,2025-06-28,290,11,4887.82,444.35,25,10,4
161,2025-06-28,290,14,6232.67,445.19,34,13,5
190,2025-06-28,290,15,7202.3,480.15,39,14,5
12,2025-06-28,290,14,4662.48,333.03,29,14,5
186,2025-06-28,290,10,6405.15,640.52,26,9,4
67,2025-06-28,290,7,2477.03,353.86,16,7,5
70,2025-06-28,290,11,5338.2,485.29,29,11,5
18,2025-06-28,290,6,3312.89,552.15,13,6,4
105,2025-06-28,290,8,4757.27,594.66,19,8,4


In [0]:
target_table = "resume_project.gold_layer.customer_features"

if spark.catalog.tableExists(target_table):
    delta_tbl = DeltaTable.forName(spark, target_table)

    (
        delta_tbl.alias("trg")
        .merge(
            cust_summary.alias("src"),
            "trg.customer_id = src.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        cust_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

### Store Level Features

In [0]:
##total revenue by store
store_summary = df.groupBy("store_id").agg(F.round(F.sum("total_amount"), 2).alias("total_revenue"))
display(store_summary)

store_id,total_revenue
5,51056.49
14,49863.52
18,56539.77
8,42122.42
16,53470.41
2,59304.63
12,53873.17
15,45462.15
19,42888.81
4,56376.69


In [0]:
##average transaction value by store
avg_trans = df.groupBy("store_id").agg(F.round(F.avg("total_amount"), 2).alias("avg_transaction_value"))

store_summary = store_summary.join(avg_trans, on="store_id", how="inner")
display(store_summary)

store_id,total_revenue,avg_transaction_value
18,56539.77,559.8
12,53873.17,476.75
16,53470.41,514.14
5,51056.49,472.75
10,53761.25,475.76
1,48645.13,540.5
3,51089.07,532.18
20,58076.84,553.11
2,59304.63,515.69
14,49863.52,547.95


In [0]:
##customer count by store
store_cust = df.groupBy("store_id").agg(F.countDistinct("customer_id").alias("customer_count"))

store_summary = store_summary.join(store_cust, on="store_id", how="inner")
display(store_summary)


store_id,total_revenue,avg_transaction_value,customer_count
18,56539.77,559.8,80
12,53873.17,476.75,89
16,53470.41,514.14,79
10,53761.25,475.76,83
5,51056.49,472.75,81
3,51089.07,532.18,72
1,48645.13,540.5,69
20,58076.84,553.11,79
2,59304.63,515.69,82
14,49863.52,547.95,72


In [0]:
##repeat customer share
cust_share = df.groupBy("store_id", "customer_id").agg(F.countDistinct("sales_id").alias("num_orders"))

cust_share = cust_share.withColumn("customer_type", F.when(F.col("num_orders") > 1, "repeat").otherwise("one_time"))

store_sum = cust_share.groupBy("store_id", "customer_type").count()
# display(store_sum)

store_totals = store_sum.groupBy("store_id").agg(F.sum("count").alias("total_customers"))
# display(store_totals)

final = store_sum.join(store_totals, on="store_id", how="inner").withColumn("cust_share_pct", F.round(F.col("count")/F.col("total_customers"), 2))
# display(final)
store_summary = store_summary.join(final, on="store_id", how="inner")
display(store_summary)

store_id,total_revenue,avg_transaction_value,customer_count,customer_type,count,total_customers,cust_share_pct
4,56376.69,490.23,88,one_time,65,88,0.74
6,53556.39,519.96,82,one_time,62,82,0.76
3,51089.07,532.18,72,one_time,53,72,0.74
10,53761.25,475.76,83,repeat,26,83,0.31
10,53761.25,475.76,83,one_time,57,83,0.69
4,56376.69,490.23,88,repeat,23,88,0.26
9,43777.92,509.05,74,one_time,64,74,0.86
16,53470.41,514.14,79,one_time,58,79,0.73
12,53873.17,476.75,89,one_time,70,89,0.79
8,42122.42,448.11,74,repeat,17,74,0.23


Setting up daily sales features for sales modeling and forecasting.

In [0]:
##dailey metrics
daily_store = df.groupBy("store_id", "date").agg(
    F.sum("total_amount").alias("daily_sales"),
    F.count("*").alias("num_transactions"),
    F.sum("quantity").alias("total_quantity"),
    F.avg("discount").alias("avg_discount")
)
display(daily_store)

store_id,date,daily_sales,num_transactions,total_quantity,avg_discount
5,2025-06-12,83.56,1,1,19.53
14,2025-03-12,83.47,1,2,21.77
18,2025-04-06,237.34,1,1,24.3
8,2025-01-06,1636.4699999999998,2,5,11.895
14,2025-02-10,1488.27,2,5,15.165
16,2024-08-08,818.76,2,4,7.91
2,2025-05-01,71.88,1,3,29.01
12,2025-06-06,903.82,1,4,24.24
15,2024-09-12,62.62,1,1,15.44
12,2024-12-02,432.2,1,1,0.01


In [0]:
##time metrics
daily_store = daily_store.withColumn("day_of_week", F.dayofweek("date")) \
    .withColumn("month", F.month("date")) \
    .withColumn("week_of_year", F.weekofyear("date"))
display(daily_store)

store_id,date,daily_sales,num_transactions,total_quantity,avg_discount,day_of_week,month,week_of_year
5,2025-06-12,83.56,1,1,19.53,5,6,24
14,2025-03-12,83.47,1,2,21.77,4,3,11
18,2025-04-06,237.34,1,1,24.3,1,4,14
8,2025-01-06,1636.4699999999998,2,5,11.895,2,1,2
14,2025-02-10,1488.27,2,5,15.165,2,2,7
16,2024-08-08,818.76,2,4,7.91,5,8,32
2,2025-05-01,71.88,1,3,29.01,5,5,18
12,2025-06-06,903.82,1,4,24.24,6,6,23
15,2024-09-12,62.62,1,1,15.44,5,9,37
12,2024-12-02,432.2,1,1,0.01,2,12,49


In [0]:
##time series features
window_spec = Window.partitionBy("store_id").orderBy("date")

daily_store = daily_store.withColumn(
    "lag_1", F.lag("daily_sales", 1).over(window_spec)
).withColumn(
    "lag_7", F.lag("daily_sales", 7).over(window_spec)

).withColumn(
    "rolling_7",
    F.avg("daily_sales").over(window_spec.rowsBetween(-7, -1))
)
display(daily_store)

store_id,date,daily_sales,num_transactions,total_quantity,avg_discount,day_of_week,month,week_of_year,lag_1,lag_7,rolling_7
1,2024-07-08,360.4,1,2,3.3,2,7,28,null,null,null
1,2024-07-11,33.57,1,1,9.76,5,7,28,360.4,null,360.4
1,2024-07-23,738.24,1,3,6.7,3,7,30,33.57,null,196.98499999999999
1,2024-07-31,1585.71,2,4,14.48,4,7,31,738.24,null,377.40333333333336
1,2024-08-01,217.38,1,3,11.87,5,8,31,1585.71,null,679.48
1,2024-08-02,277.72,1,1,11.42,6,8,31,217.38,null,587.0600000000001
1,2024-08-09,81.75,1,2,16.24,6,8,32,277.72,null,535.5033333333334
1,2024-08-14,311.65,1,2,18.16,4,8,33,81.75,360.4,470.6814285714286
1,2024-08-27,157.71,1,1,27.2,3,8,35,311.65,33.57,463.71714285714285
1,2024-08-30,1283.02,1,4,18.52,6,8,35,157.71,738.24,481.4514285714286


In [0]:
store_summary_wide = (
    store_summary
    .groupBy("store_id", "total_revenue", "avg_transaction_value", "customer_count", "total_customers")
    .pivot("customer_type", ["one_time", "repeat"])
    .agg(
        F.first("count").alias("count"),
        F.first("cust_share_pct").alias("share_pct")
    )
)
display(store_summary_wide)

store_id,total_revenue,avg_transaction_value,customer_count,total_customers,one_time_count,one_time_share_pct,repeat_count,repeat_share_pct
11,37098.39,446.97,73,73,64,0.88,9,0.12
10,53761.25,475.76,83,83,57,0.69,26,0.31
2,59304.63,515.69,82,82,57,0.7,25,0.3
4,56376.69,490.23,88,88,65,0.74,23,0.26
15,45462.15,478.55,75,75,57,0.76,18,0.24
20,58076.84,553.11,79,79,57,0.72,22,0.28
6,53556.39,519.96,82,82,62,0.76,20,0.24
9,43777.92,509.05,74,74,64,0.86,10,0.14
17,44738.98,475.95,71,71,54,0.76,17,0.24
3,51089.07,532.18,72,72,53,0.74,19,0.26


In [0]:
daily_store_final = daily_store.join(
    store_summary_wide,
    on="store_id",
    how="left"
)
display(daily_store_final)

store_id,date,daily_sales,num_transactions,total_quantity,avg_discount,day_of_week,month,week_of_year,lag_1,lag_7,rolling_7,total_revenue,avg_transaction_value,customer_count,total_customers,one_time_count,one_time_share_pct,repeat_count,repeat_share_pct
1,2024-07-08,360.4,1,2,3.3,2,7,28,null,null,null,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-07-11,33.57,1,1,9.76,5,7,28,360.4,null,360.4,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-07-23,738.24,1,3,6.7,3,7,30,33.57,null,196.98499999999999,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-07-31,1585.71,2,4,14.48,4,7,31,738.24,null,377.40333333333336,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-01,217.38,1,3,11.87,5,8,31,1585.71,null,679.48,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-02,277.72,1,1,11.42,6,8,31,217.38,null,587.0600000000001,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-09,81.75,1,2,16.24,6,8,32,277.72,null,535.5033333333334,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-14,311.65,1,2,18.16,4,8,33,81.75,360.4,470.6814285714286,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-27,157.71,1,1,27.2,3,8,35,311.65,33.57,463.71714285714285,48645.13,540.5,69,69,51,0.74,18,0.26
1,2024-08-30,1283.02,1,4,18.52,6,8,35,157.71,738.24,481.4514285714286,48645.13,540.5,69,69,51,0.74,18,0.26


In [0]:
target_table = "resume_project.gold_layer.store_features"

if spark.catalog.tableExists(target_table):
    delta_tbl = DeltaTable.forName(spark, target_table)

    (
        delta_tbl.alias("trg")
        .merge(
            daily_store_final.alias("src"),
            "trg.store_id = src.store_id AND trg.date = src.date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        daily_store_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

### Product Level Features

In [0]:
##total quantity by product
prod_summary = df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity"))
display(prod_summary)

product_id,total_quantity
27,43
38,63
82,77
36,43
21,41
35,40
7,52
29,76
22,44
81,59


In [0]:
##total revenue by product
total_rev_prod = df.groupBy("product_id").agg(F.round(F.sum("total_amount"), 2).alias("total_revenue"))
# display(total_rev_prod)

prod_summary = prod_summary.join(total_rev_prod, on="product_id", how="inner")
display(prod_summary)

product_id,total_quantity,total_revenue
38,63,2772.12
93,57,17861.36
67,45,2801.31
18,68,15159.86
12,50,19325.64
70,39,15890.18
94,48,11066.98
74,53,18907.61
16,36,2904.17
64,52,7854.49


In [0]:
##average revenue per product
avg_rev_prod = df.groupBy("product_id").agg(F.round(F.avg("total_amount"), 2).alias("avg_revenue_per_transaction"))
# display(avg_rev_prod)

prod_summary = prod_summary.join(avg_rev_prod, on="product_id", how="inner")
display(prod_summary)

product_id,total_quantity,total_revenue,avg_revenue_per_transaction
38,63,2772.12,110.88
93,57,17861.36,776.58
67,45,2801.31,155.63
18,68,15159.86,522.75
12,50,19325.64,1073.65
70,39,15890.18,882.79
94,48,11066.98,553.35
74,53,18907.61,945.38
16,36,2904.17,170.83
64,52,7854.49,392.72


In [0]:
target_table = "resume_project.gold_layer.product_features"

if spark.catalog.tableExists(target_table):
    delta_tbl = DeltaTable.forName(spark, target_table)

    (
        delta_tbl.alias("trg")
        .merge(
            prod_summary.alias("src"),
            "trg.product_id = src.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        prod_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )